# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR<sup>2</sup> mass spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library for FAIR data.

### Dataset Source
This dataset is described using the [Croissant schema](https://mlcommons.org/croissant/) and is accessible via the following URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

- The Croissant JSON-LD file is located here:
    - https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Dataset Description:")
print(metadata.description)


## 2. Data Overview
Review available record sets, including their `@id` fields, along with the fields and columns inside each record set.

In Croissant, each data table is a 'record set' accessible by its `@id`.

**List all record sets and their fields:**

In [ ]:
# Show all record sets and fields
record_sets = metadata.record_sets
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id} | name: {getattr(rs, 'name', 'N/A')}")
    print("  Fields:")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    - Field @id: {field.id} | name: {getattr(field, 'name', 'N/A')} | dataType: {getattr(field, 'data_type', 'N/A')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

First, select one or more record sets (using their `@id`) for further analysis. Here, we demonstrate with the main protein data record set.


In [ ]:
# The main record set in this dataset appears to be protein data. Find its @id:
main_record_set_id = None
for rs in metadata.record_sets:
    # Heuristically select first tabular record set (most have 'Protein' in the name)
    name_lower = getattr(rs, 'name', '').lower()
    if 'protein' in name_lower or 'data' in name_lower or True:  # dataset has usually one main table
        main_record_set_id = rs.id
        break

if main_record_set_id is None:
    raise RuntimeError("No record set found for extraction.")

print(f"Loading data from RecordSet @id: {main_record_set_id}")

# Load all available record sets into dataframes (using @id)
df_dict = {}
for rs in metadata.record_sets:
    records = list(dataset.records(record_set=rs.id))
    if records:
        df_dict[rs.id] = pd.DataFrame(records)
    else:
        df_dict[rs.id] = pd.DataFrame()

# Display columns of the main record set DataFrame
print(f"Columns in DataFrame for RecordSet @id {main_record_set_id}:")
print(df_dict[main_record_set_id].columns.tolist())

df_dict[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping.

We select numeric fields for filtering and normalization, e.g. `Coverage (%)` or `MW [kDa]`, referenced by their `@id` field in the Croissant schema.

In [ ]:
# Choose a numeric field for EDA. Find numeric fields from record set fields.
rs = None
for r in metadata.record_sets:
    if r.id == main_record_set_id:
        rs = r
        break
assert rs is not None
# Find first field of type Float/Integer
numeric_field = None
numeric_id = None
for f in rs.fields:
    if getattr(f, 'data_type', '').lower() in ['float', 'number', 'double', 'integer'] or 'percent' in getattr(f, 'name', '').lower() or 'mw' in getattr(f, 'name', '').lower():
        numeric_field = getattr(f, 'name', f.id)
        numeric_id = f.id
        break
if numeric_id is None:
    raise RuntimeError("No numeric field found in main record set.")
print(f"Using numeric field: {numeric_field} (@id: {numeric_id})")

# Filtering records example: keep rows where the numeric field > threshold
threshold = 10
df = df_dict[main_record_set_id]

# Some columns may have non-numeric types, so coerce to numeric, errors='coerce' will insert NaN for bad data
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with '{numeric_field}' > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if available
group_field = None
category_candidates = [f for f in rs.fields if getattr(f, 'data_type', '').lower() in ['string', 'text'] and f.id != numeric_id]
if category_candidates:
    group_field = getattr(category_candidates[0], 'name', category_candidates[0].id)

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped average '{numeric_field}' by '{group_field}' (")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions (e.g., histogram of the selected numeric field) or relationships between fields.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (filtered > {threshold})")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If group field was found, visualize group means
if group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(10,4))
    (filtered_df.groupby(group_field)[numeric_field]
        .mean()
        .sort_values(ascending=False)
        .head(20)
        .plot(kind='bar', color='cornflowerblue'))
    plt.title(f'Mean {numeric_field} by {group_field} (top 20)')
    plt.ylabel(f'Mean {numeric_field}')
    plt.xlabel(group_field)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load Croissant metadata and records using `mlcroissant`
- Enumerate and inspect record sets and field definitions by `@id`
- Load and process tabular data with field references by `@id`
- Apply basic exploratory analysis and visualization

Proceed to further biological or statistical analysis as appropriate for your research questions!
